In [1]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 },
 "cells": [
  {
   "cell_type": "markdown",
   "id": "a1b2c3d4",
   "metadata": {},
   "source": [
    "# Twitter Sentiment Analysis\n",
    "**Stack:** Python, PyTorch, BERT, HuggingFace Transformers, NLTK, pandas, scikit-learn, Plotly\n",
    "\n",
    "This notebook applies LLM-based NLP methods to unstructured Twitter text data. We fine-tune BERT for sentiment classification on 50K tweets from the Sentiment140 dataset, covering full preprocessing, training, evaluation, and visualization pipelines."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b2c3d4e5",
   "metadata": {},
   "source": [
    "## 1. Install & Import Dependencies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c3d4e5f6",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run once to install required packages\n",
    "import subprocess, sys\n",
    "packages = [\n",
    "    'transformers', 'torch', 'nltk', 'pandas', 'scikit-learn',\n",
    "    'plotly', 'datasets', 'accelerate', 'tqdm'\n",
    "]\n",
    "for pkg in packages:\n",
    "    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])\n",
    "print('All packages installed.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d4e5f6a7",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import re\n",
    "import os\n",
    "import torch\n",
    "import nltk\n",
    "import plotly.express as px\n",
    "import plotly.graph_objects as go\n",
    "from plotly.subplots import make_subplots\n",
    "\n",
    "from nltk.corpus import stopwords\n",
    "from nltk.tokenize import word_tokenize\n",
    "from nltk.stem import WordNetLemmatizer\n",
    "\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.metrics import (\n",
    "    accuracy_score, classification_report,\n",
    "    confusion_matrix, ConfusionMatrixDisplay\n",
    ")\n",
    "\n",
    "from transformers import (\n",
    "    BertTokenizerFast,\n",
    "    BertForSequenceClassification,\n",
    "    Trainer,\n",
    "    TrainingArguments,\n",
    "    EarlyStoppingCallback\n",
    ")\n",
    "from torch.utils.data import Dataset\n",
    "from tqdm.auto import tqdm\n",
    "\n",
    "# NLTK downloads\n",
    "for resource in ['stopwords', 'punkt', 'wordnet', 'omw-1.4', 'punkt_tab']:\n",
    "    nltk.download(resource, quiet=True)\n",
    "\n",
    "# Device config\n",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
    "print(f'Using device: {device}')\n",
    "\n",
    "SEED = 42\n",
    "np.random.seed(SEED)\n",
    "torch.manual_seed(SEED)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e5f6a7b8",
   "metadata": {},
   "source": [
    "## 2. Load & Sample Dataset\n",
    "We use the **Sentiment140** dataset (1.6M tweets). We sample **50,000 tweets** (25K positive, 25K negative) for balanced training."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f6a7b8c9",
   "metadata": {},
   "outputs": [],
   "source": [
    "from datasets import load_dataset\n",
    "\n",
    "print('Loading Sentiment140 dataset...')\n",
    "raw = load_dataset('sentiment140', trust_remote_code=True)\n",
    "df_full = pd.DataFrame(raw['train'])\n",
    "\n",
    "# Rename columns\n",
    "df_full.columns = ['sentiment', 'id', 'date', 'query', 'user', 'text']\n",
    "\n",
    "# Sentiment140 uses 0=negative, 4=positive — remap to 0/1\n",
    "df_full['label'] = df_full['sentiment'].map({0: 0, 4: 1})\n",
    "df_full = df_full[['text', 'label']].dropna()\n",
    "\n",
    "# Sample 25K per class for balanced 50K dataset\n",
    "df = pd.concat([\n",
    "    df_full[df_full['label'] == 0].sample(25000, random_state=SEED),\n",
    "    df_full[df_full['label'] == 1].sample(25000, random_state=SEED)\n",
    "]).sample(frac=1, random_state=SEED).reset_index(drop=True)\n",
    "\n",
    "print(f'Dataset shape: {df.shape}')\n",
    "print(df['label'].value_counts())\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a7b8c9d0",
   "metadata": {},
   "source": [
    "## 3. Exploratory Data Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b8c9d0e1",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Label distribution\n",
    "label_counts = df['label'].value_counts().reset_index()\n",
    "label_counts.columns = ['label', 'count']\n",
    "label_counts['sentiment'] = label_counts['label'].map({0: 'Negative', 1: 'Positive'})\n",
    "\n",
    "fig = px.bar(\n",
    "    label_counts, x='sentiment', y='count',\n",
    "    color='sentiment',\n",
    "    color_discrete_map={'Positive': '#2ecc71', 'Negative': '#e74c3c'},\n",
    "    title='Label Distribution in 50K Sample',\n",
    "    text='count'\n",
    ")\n",
    "fig.update_traces(textposition='outside')\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c9d0e1f2",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Tweet length distribution\n",
    "df['tweet_length'] = df['text'].apply(len)\n",
    "\n",
    "fig = px.histogram(\n",
    "    df, x='tweet_length', color=df['label'].map({0:'Negative', 1:'Positive'}),\n",
    "    barmode='overlay', nbins=60,\n",
    "    title='Tweet Character Length Distribution by Sentiment',\n",
    "    labels={'tweet_length': 'Character Count', 'color': 'Sentiment'},\n",
    "    color_discrete_map={'Positive': '#2ecc71', 'Negative': '#e74c3c'},\n",
    "    opacity=0.7\n",
    ")\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d0e1f2a3",
   "metadata": {},
   "source": [
    "## 4. Text Preprocessing (NLTK)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e1f2a3b4",
   "metadata": {},
   "outputs": [],
   "source": [
    "stop_words = set(stopwords.words('english'))\n",
    "lemmatizer = WordNetLemmatizer()\n",
    "\n",
    "def clean_tweet(text):\n",
    "    \"\"\"Full preprocessing pipeline for raw tweet text.\"\"\"\n",
    "    text = str(text).lower()\n",
    "    text = re.sub(r'http\\S+|www\\S+', '', text)         # remove URLs\n",
    "    text = re.sub(r'@\\w+', '', text)                    # remove mentions\n",
    "    text = re.sub(r'#(\\w+)', r'\\1', text)               # strip hashtag symbol\n",
    "    text = re.sub(r'[^a-z\\s]', '', text)                # remove non-alpha\n",
    "    text = re.sub(r'\\s+', ' ', text).strip()            # normalize whitespace\n",
    "    tokens = word_tokenize(text)\n",
    "    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]\n",
    "    return ' '.join(tokens)\n",
    "\n",
    "tqdm.pandas(desc='Cleaning tweets')\n",
    "df['clean_text'] = df['text'].progress_apply(clean_tweet)\n",
    "\n",
    "# Remove empty rows after cleaning\n",
    "df = df[df['clean_text'].str.strip().astype(bool)].reset_index(drop=True)\n",
    "print(f'After cleaning: {df.shape}')\n",
    "df[['text', 'clean_text', 'label']].head()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f2a3b4c5",
   "metadata": {},
   "source": [
    "## 5. Train / Validation / Test Split"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a3b4c5d6",
   "metadata": {},
   "outputs": [],
   "source": [
    "X = df['clean_text'].tolist()\n",
    "y = df['label'].tolist()\n",
    "\n",
    "X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)\n",
    "X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)\n",
    "\n",
    "print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b4c5d6e7",
   "metadata": {},
   "source": [
    "## 6. Tokenization & Dataset Class (HuggingFace / BERT)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c5d6e7f8",
   "metadata": {},
   "outputs": [],
   "source": [
    "MODEL_NAME = 'bert-base-uncased'\n",
    "tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)\n",
    "\n",
    "class TweetDataset(Dataset):\n",
    "    def __init__(self, texts, labels, tokenizer, max_len=128):\n",
    "        self.encodings = tokenizer(\n",
    "            texts,\n",
    "            truncation=True,\n",
    "            padding='max_length',\n",
    "            max_length=max_len,\n",
    "            return_tensors='pt'\n",
    "        )\n",
    "        self.labels = torch.tensor(labels, dtype=torch.long)\n",
    "\n",
    "    def __len__(self):\n",
    "        return len(self.labels)\n",
    "\n",
    "    def __getitem__(self, idx):\n",
    "        return {\n",
    "            'input_ids':      self.encodings['input_ids'][idx],\n",
    "            'attention_mask': self.encodings['attention_mask'][idx],\n",
    "            'token_type_ids': self.encodings['token_type_ids'][idx],\n",
    "            'labels':         self.labels[idx]\n",
    "        }\n",
    "\n",
    "print('Tokenizing splits...')\n",
    "train_dataset = TweetDataset(X_train, y_train, tokenizer)\n",
    "val_dataset   = TweetDataset(X_val,   y_val,   tokenizer)\n",
    "test_dataset  = TweetDataset(X_test,  y_test,  tokenizer)\n",
    "print('Tokenization complete.')"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d6e7f8a9",
   "metadata": {},
   "source": [
    "## 7. Fine-Tune BERT for Sentiment Classification"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e7f8a9b0",
   "metadata": {},
   "outputs": [],
   "source": [
    "model = BertForSequenceClassification.from_pretrained(\n",
    "    MODEL_NAME,\n",
    "    num_labels=2,\n",
    "    id2label={0: 'NEGATIVE', 1: 'POSITIVE'},\n",
    "    label2id={'NEGATIVE': 0, 'POSITIVE': 1}\n",
    ")\n",
    "model.to(device)\n",
    "print('Model loaded.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f8a9b0c1",
   "metadata": {},
   "outputs": [],
   "source": [
    "def compute_metrics(eval_pred):\n",
    "    logits, labels = eval_pred\n",
    "    preds = np.argmax(logits, axis=-1)\n",
    "    acc = accuracy_score(labels, preds)\n",
    "    report = classification_report(labels, preds, output_dict=True, zero_division=0)\n",
    "    return {\n",
    "        'accuracy':  acc,\n",
    "        'precision': report['weighted avg']['precision'],\n",
    "        'recall':    report['weighted avg']['recall'],\n",
    "        'f1':        report['weighted avg']['f1-score']\n",
    "    }\n",
    "\n",
    "training_args = TrainingArguments(\n",
    "    output_dir='./bert_sentiment',\n",
    "    num_train_epochs=3,\n",
    "    per_device_train_batch_size=32,\n",
    "    per_device_eval_batch_size=64,\n",
    "    warmup_steps=200,\n",
    "    weight_decay=0.01,\n",
    "    learning_rate=2e-5,\n",
    "    eval_strategy='epoch',\n",
    "    save_strategy='epoch',\n",
    "    load_best_model_at_end=True,\n",
    "    metric_for_best_model='accuracy',\n",
    "    logging_dir='./logs',\n",
    "    logging_steps=100,\n",
    "    seed=SEED,\n",
    "    fp16=torch.cuda.is_available(),\n",
    "    report_to='none'\n",
    ")\n",
    "\n",
    "trainer = Trainer(\n",
    "    model=model,\n",
    "    args=training_args,\n",
    "    train_dataset=train_dataset,\n",
    "    eval_dataset=val_dataset,\n",
    "    compute_metrics=compute_metrics,\n",
    "    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]\n",
    ")\n",
    "\n",
    "print('Starting fine-tuning...')\n",
    "trainer.train()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a9b0c1d2",
   "metadata": {},
   "source": [
    "## 8. Evaluation on Test Set"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b0c1d2e3",
   "metadata": {},
   "outputs": [],
   "source": [
    "print('Evaluating on test set...')\n",
    "preds_output = trainer.predict(test_dataset)\n",
    "y_pred = np.argmax(preds_output.predictions, axis=-1)\n",
    "\n",
    "acc = accuracy_score(y_test, y_pred)\n",
    "print(f'\\nTest Accuracy: {acc:.4f} ({acc*100:.2f}%)')\n",
    "print('\\nClassification Report:')\n",
    "print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c1d2e3f4",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Training history visualization\n",
    "log_history = trainer.state.log_history\n",
    "train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]\n",
    "eval_logs  = [l for l in log_history if 'eval_accuracy' in l]\n",
    "\n",
    "fig = make_subplots(rows=1, cols=2, subplot_titles=('Training Loss', 'Validation Accuracy per Epoch'))\n",
    "\n",
    "fig.add_trace(go.Scatter(\n",
    "    x=[l['step'] for l in train_logs],\n",
    "    y=[l['loss'] for l in train_logs],\n",
    "    mode='lines', name='Train Loss', line=dict(color='#e74c3c')\n",
    "), row=1, col=1)\n",
    "\n",
    "fig.add_trace(go.Scatter(\n",
    "    x=[l['epoch'] for l in eval_logs],\n",
    "    y=[l['eval_accuracy'] for l in eval_logs],\n",
    "    mode='lines+markers', name='Val Accuracy', line=dict(color='#2ecc71')\n",
    "), row=1, col=2)\n",
    "\n",
    "fig.update_layout(title='BERT Fine-Tuning: Training Metrics', height=400)\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d2e3f4a5",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Confusion matrix\n",
    "cm = confusion_matrix(y_test, y_pred)\n",
    "fig = px.imshow(\n",
    "    cm,\n",
    "    labels=dict(x='Predicted', y='Actual', color='Count'),\n",
    "    x=['Negative', 'Positive'],\n",
    "    y=['Negative', 'Positive'],\n",
    "    color_continuous_scale='Blues',\n",
    "    title='Confusion Matrix — BERT Sentiment Classifier',\n",
    "    text_auto=True\n",
    ")\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e3f4a5b6",
   "metadata": {},
   "source": [
    "## 9. Save Model & Tokenizer"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f4a5b6c7",
   "metadata": {},
   "outputs": [],
   "source": [
    "SAVE_DIR = './bert_sentiment_final'\n",
    "model.save_pretrained(SAVE_DIR)\n",
    "tokenizer.save_pretrained(SAVE_DIR)\n",
    "print(f'Model and tokenizer saved to {SAVE_DIR}')"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a5b6c7d8",
   "metadata": {},
   "source": [
    "## 10. Inference — Run on New Tweets"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b6c7d8e9",
   "metadata": {},
   "outputs": [],
   "source": [
    "from transformers import pipeline\n",
    "\n",
    "sentiment_pipeline = pipeline(\n",
    "    'text-classification',\n",
    "    model=SAVE_DIR,\n",
    "    tokenizer=SAVE_DIR,\n",
    "    device=0 if torch.cuda.is_available() else -1\n",
    ")\n",
    "\n",
    "sample_tweets = [\n",
    "    \"I absolutely love this product, it's amazing!\",\n",
    "    \"Worst experience ever, totally disappointed.\",\n",
    "    \"The weather today is just okay, nothing special.\",\n",
    "    \"So excited for the weekend plans ahead!\",\n",
    "    \"Can't believe how bad the service was today.\"\n",
    "]\n",
    "\n",
    "results = sentiment_pipeline(sample_tweets)\n",
    "results_df = pd.DataFrame({\n",
    "    'tweet': sample_tweets,\n",
    "    'prediction': [r['label'] for r in results],\n",
    "    'confidence': [round(r['score'], 4) for r in results]\n",
    "})\n",
    "print(results_df.to_string(index=False))"
   ]
  }
 ]
}

NameError: name 'null' is not defined